# Chapter 12.5: Hardware-Aware Optimization -- AMD, Intel, Custom ASICs

## Overview

LLM inference is expanding beyond NVIDIA GPUs. Understanding hardware-specific characteristics is crucial for optimal serving across AMD, Intel, Google TPU, and custom ASICs.

### Hardware Landscape
| Hardware | Compute (FP16 TFLOPS) | Memory BW (GB/s) | Memory (GB) | Software Stack |
|----------|----------------------|-------------------|-------------|----------------|
| NVIDIA H100 SXM | 989 | 3,350 | 80 | CUDA / cuDNN |
| NVIDIA A100 SXM | 312 | 2,039 | 80 | CUDA / cuDNN |
| AMD MI300X | 1,307 | 5,300 | 192 | ROCm / hipBLAS |
| Intel Gaudi 2 | 432 | 2,460 | 96 | Habana SynapseAI |
| Google TPU v5e | ~197 | ~819 | 16 (HBM) | JAX / XLA |
| AWS Inferentia2 | ~190 | ~820 | 32 | Neuron SDK |

### Learning Objectives
1. Compare hardware characteristics for LLM inference
2. Profile memory bandwidth bottlenecks
3. Analyze hardware-specific kernel optimization opportunities
4. Build a hardware recommendation engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part12/chapter_12.5_hardware.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part12/chapter_12.5_hardware.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
import json

np.random.seed(42)
plt.style.use('default')
print("Hardware-aware optimization environment ready.")

In [ ]:
@dataclass
class HardwareSpec:
    name: str
    vendor: str
    compute_fp16_tflops: float
    compute_fp8_tflops: float  # 0 if not supported
    memory_gb: float
    memory_bandwidth_gbps: float
    interconnect_bandwidth_gbps: float  # GPU-to-GPU
    tdp_watts: float
    price_per_hour: float  # Cloud pricing estimate
    software_stack: str
    maturity: str  # 'mature', 'developing', 'early'

    @property
    def compute_intensity_threshold(self):
        """Ops/byte at which we transition from memory-bound to compute-bound."""
        return self.compute_fp16_tflops * 1e12 / (self.memory_bandwidth_gbps * 1e9)

    @property
    def perf_per_dollar(self):
        return self.compute_fp16_tflops / self.price_per_hour

    @property
    def perf_per_watt(self):
        return self.compute_fp16_tflops / self.tdp_watts * 1000


# Hardware database
hardware_db = {
    'H100_SXM': HardwareSpec('H100 SXM', 'NVIDIA', 989, 1979, 80, 3350, 900, 700, 3.50, 'CUDA', 'mature'),
    'H200': HardwareSpec('H200', 'NVIDIA', 989, 1979, 141, 4800, 900, 700, 4.50, 'CUDA', 'mature'),
    'A100_SXM': HardwareSpec('A100 SXM', 'NVIDIA', 312, 0, 80, 2039, 600, 400, 2.20, 'CUDA', 'mature'),
    'L40S': HardwareSpec('L40S', 'NVIDIA', 362, 724, 48, 864, 0, 350, 1.50, 'CUDA', 'mature'),
    'MI300X': HardwareSpec('MI300X', 'AMD', 1307, 2614, 192, 5300, 896, 750, 3.00, 'ROCm', 'developing'),
    'MI250X': HardwareSpec('MI250X', 'AMD', 383, 0, 128, 3277, 800, 560, 1.80, 'ROCm', 'developing'),
    'Gaudi2': HardwareSpec('Gaudi 2', 'Intel', 432, 865, 96, 2460, 300, 600, 1.80, 'SynapseAI', 'developing'),
    'Gaudi3': HardwareSpec('Gaudi 3', 'Intel', 1835, 3670, 128, 3700, 600, 900, 3.00, 'SynapseAI', 'early'),
    'TPUv5e': HardwareSpec('TPU v5e', 'Google', 197, 394, 16, 819, 1600, 170, 1.20, 'JAX/XLA', 'mature'),
    'Inferentia2': HardwareSpec('Inferentia2', 'AWS', 190, 380, 32, 820, 384, 185, 0.75, 'Neuron', 'developing'),
}

print(f"{'Hardware':>15} {'Vendor':>8} {'FP16 TF':>8} {'Mem GB':>7} {'BW GB/s':>8} "
      f"{'AI Thresh':>10} {'$/hr':>6} {'Perf/$':>7} {'Stack':>12}")
print("-" * 95)
for name, hw in hardware_db.items():
    print(f"{hw.name:>15} {hw.vendor:>8} {hw.compute_fp16_tflops:>7.0f} {hw.memory_gb:>6.0f} "
          f"{hw.memory_bandwidth_gbps:>7.0f} {hw.compute_intensity_threshold:>9.0f} "
          f"${hw.price_per_hour:>5.2f} {hw.perf_per_dollar:>6.0f} {hw.software_stack:>12}")

## 2. Demo: Compare Kernel Performance Across Hardware

In [ ]:
class KernelPerformanceModel:
    """Model kernel performance on different hardware."""

    def __init__(self, hw: HardwareSpec):
        self.hw = hw

    def gemm_time(self, m: int, n: int, k: int, dtype_bytes: int = 2) -> float:
        """Estimate GEMM (matrix multiply) execution time."""
        flops = 2 * m * n * k
        # GEMM is compute-bound for large matrices
        bytes_accessed = (m * k + k * n + m * n) * dtype_bytes
        ai = flops / bytes_accessed

        compute_time = flops / (self.hw.compute_fp16_tflops * 1e12)
        memory_time = bytes_accessed / (self.hw.memory_bandwidth_gbps * 1e9)
        return max(compute_time, memory_time)

    def attention_time(self, batch: int, seq_len: int, heads: int,
                       head_dim: int, dtype_bytes: int = 2) -> float:
        """Estimate attention kernel time."""
        # QK^T: [batch*heads, seq, head_dim] x [batch*heads, head_dim, seq]
        qk_flops = 2 * batch * heads * seq_len * seq_len * head_dim
        # Softmax: ~5 ops per element
        softmax_flops = 5 * batch * heads * seq_len * seq_len
        # AV: [batch*heads, seq, seq] x [batch*heads, seq, head_dim]
        av_flops = 2 * batch * heads * seq_len * seq_len * head_dim

        total_flops = qk_flops + softmax_flops + av_flops
        # Memory: load Q, K, V, write O
        mem_bytes = batch * heads * seq_len * head_dim * dtype_bytes * 4
        # FlashAttention: only load/store QKV+O, not full attention matrix
        # Without FlashAttention: also store seq_len^2 attention matrix

        compute_time = total_flops / (self.hw.compute_fp16_tflops * 1e12)
        memory_time = mem_bytes / (self.hw.memory_bandwidth_gbps * 1e9)
        return max(compute_time, memory_time)

    def decode_step_time(self, batch: int, num_layers: int, hidden_dim: int,
                          dtype_bytes: int = 2) -> float:
        """Estimate single decode step time (memory-bound)."""
        # Load all model weights per decode step
        total_params = 12 * num_layers * hidden_dim ** 2
        bytes_to_load = total_params * dtype_bytes
        # FLOPs: 2 * params * batch_size
        flops = 2 * total_params * batch

        compute_time = flops / (self.hw.compute_fp16_tflops * 1e12)
        memory_time = bytes_to_load / (self.hw.memory_bandwidth_gbps * 1e9)
        return max(compute_time, memory_time)


# Compare all hardware
model_configs = [
    ('7B', 32, 4096),
    ('13B', 40, 5120),
    ('70B', 80, 8192),
]

print(f"Decode TPOT (ms) per token, batch_size=1:")
print(f"{'Hardware':>15}", end='')
for name, _, _ in model_configs:
    print(f"{name:>12}", end='')
print()
print("-" * (15 + 12 * len(model_configs)))

for hw_name, hw in hardware_db.items():
    perf = KernelPerformanceModel(hw)
    print(f"{hw.name:>15}", end='')
    for name, layers, hidden in model_configs:
        t = perf.decode_step_time(1, layers, hidden) * 1000
        print(f"{t:>10.1f}ms", end='')
    print()

print(f"\nDecode TPOT (ms) per token, batch_size=32:")
print(f"{'Hardware':>15}", end='')
for name, _, _ in model_configs:
    print(f"{name:>12}", end='')
print()
print("-" * (15 + 12 * len(model_configs)))
for hw_name, hw in hardware_db.items():
    perf = KernelPerformanceModel(hw)
    print(f"{hw.name:>15}", end='')
    for name, layers, hidden in model_configs:
        t = perf.decode_step_time(32, layers, hidden) * 1000
        print(f"{t:>10.1f}ms", end='')
    print()

In [ ]:
# Visualize hardware comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

hw_names = [hw.name for hw in hardware_db.values()]
colors = {'NVIDIA': 'green', 'AMD': 'red', 'Intel': 'blue', 'Google': 'orange', 'AWS': 'purple'}
bar_colors = [colors[hw.vendor] for hw in hardware_db.values()]

# Roofline comparison
ax = axes[0, 0]
for hw in hardware_db.values():
    ai_range = np.logspace(-1, 4, 200)
    perf = np.minimum(hw.compute_fp16_tflops, hw.memory_bandwidth_gbps * 1e-3 * ai_range)
    ax.loglog(ai_range, perf, linewidth=2, label=hw.name, color=colors[hw.vendor], alpha=0.7)
ax.set_xlabel('Arithmetic Intensity (FLOPs/Byte)')
ax.set_ylabel('Performance (TFLOPS)')
ax.set_title('Roofline Comparison')
ax.legend(fontsize=6, ncol=2)
ax.grid(True, alpha=0.3)

# Memory bandwidth comparison
ax = axes[0, 1]
x = np.arange(len(hw_names))
ax.barh(x, [hw.memory_bandwidth_gbps for hw in hardware_db.values()], color=bar_colors)
ax.set_yticks(x)
ax.set_yticklabels(hw_names, fontsize=8)
ax.set_xlabel('Memory Bandwidth (GB/s)')
ax.set_title('Memory Bandwidth (key for decode)')

# Performance per dollar
ax = axes[1, 0]
ax.barh(x, [hw.perf_per_dollar for hw in hardware_db.values()], color=bar_colors)
ax.set_yticks(x)
ax.set_yticklabels(hw_names, fontsize=8)
ax.set_xlabel('FP16 TFLOPS per $/hour')
ax.set_title('Performance per Dollar')

# Memory capacity
ax = axes[1, 1]
ax.barh(x, [hw.memory_gb for hw in hardware_db.values()], color=bar_colors)
ax.set_yticks(x)
ax.set_yticklabels(hw_names, fontsize=8)
ax.set_xlabel('Memory (GB)')
ax.set_title('Memory Capacity')

plt.tight_layout()
plt.savefig('hardware_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Demo: Profile Memory Bandwidth on Different GPUs

In [ ]:
class MemoryBandwidthProfiler:
    """Profile effective memory bandwidth utilization for LLM inference."""

    def __init__(self, hw: HardwareSpec):
        self.hw = hw

    def profile_decode(self, model_size_gb: float, batch_sizes: List[int]) -> List[dict]:
        """Profile bandwidth utilization during decode phase."""
        results = []
        for bs in batch_sizes:
            # Decode: load all weights, compute batch_size tokens
            bytes_loaded = model_size_gb * 1e9  # Model weights
            flops = bytes_loaded * bs  # ~2 FLOPs per byte per token

            # Effective bandwidth utilization depends on batch size
            # Small batch: memory-bound, high BW utilization
            # Large batch: compute-bound, lower BW utilization (but higher compute)
            memory_time = bytes_loaded / (self.hw.memory_bandwidth_gbps * 1e9)
            compute_time = flops / (self.hw.compute_fp16_tflops * 1e12)

            actual_time = max(memory_time, compute_time)
            effective_bw = bytes_loaded / actual_time / 1e9
            bw_utilization = effective_bw / self.hw.memory_bandwidth_gbps

            # Account for overhead (kernel launch, synchronization)
            overhead_factor = 0.85  # Realistic efficiency
            effective_bw *= overhead_factor

            results.append({
                'batch_size': bs,
                'effective_bw_gbps': effective_bw,
                'bw_utilization': bw_utilization * overhead_factor,
                'compute_utilization': min(1.0, compute_time / actual_time) * overhead_factor,
                'bottleneck': 'memory' if memory_time > compute_time else 'compute',
                'tpot_ms': actual_time * 1000 / overhead_factor,
                'tokens_per_second': bs / actual_time * overhead_factor,
            })
        return results

    def find_optimal_batch_size(self, model_size_gb: float) -> int:
        """Find batch size that maximizes throughput."""
        batch_sizes = list(range(1, 257))
        results = self.profile_decode(model_size_gb, batch_sizes)
        best = max(results, key=lambda r: r['tokens_per_second'])
        return best['batch_size']


# Profile all hardware for LLaMA-70B
model_size = 140  # GB in FP16
batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256]

print(f"Decode Performance Profile - LLaMA-70B (FP16):")
print(f"{'Hardware':>15} {'Opt BS':>7} {'BW Util':>8} {'Peak tok/s':>10} {'TPOT@bs1':>10}")
print("-" * 55)

hw_profiles = {}
for name, hw in hardware_db.items():
    if hw.memory_gb < model_size / 2:  # Can't fit model
        continue
    profiler = MemoryBandwidthProfiler(hw)
    results = profiler.profile_decode(model_size, batch_sizes)
    hw_profiles[name] = results

    opt_bs = profiler.find_optimal_batch_size(model_size)
    opt_result = [r for r in results if r['batch_size'] == opt_bs][0]
    bs1_result = results[0]

    print(f"{hw.name:>15} {opt_bs:>7} {opt_result['bw_utilization']:>7.0%} "
          f"{opt_result['tokens_per_second']:>9.0f}/s {bs1_result['tpot_ms']:>8.1f}ms")

In [ ]:
# Visualize bandwidth utilization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for name, results in hw_profiles.items():
    hw = hardware_db[name]
    bs_vals = [r['batch_size'] for r in results]

    # Throughput vs batch size
    tps = [r['tokens_per_second'] for r in results]
    axes[0].plot(bs_vals, tps, 'o-', label=hw.name, color=colors[hw.vendor])

    # BW utilization vs batch size
    bw_util = [r['bw_utilization'] for r in results]
    axes[1].plot(bs_vals, bw_util, 'o-', label=hw.name, color=colors[hw.vendor])

    # TPOT vs batch size
    tpot = [r['tpot_ms'] for r in results]
    axes[2].plot(bs_vals, tpot, 'o-', label=hw.name, color=colors[hw.vendor])

axes[0].set_xlabel('Batch Size')
axes[0].set_ylabel('Tokens/second')
axes[0].set_title('Throughput vs Batch Size')
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log', base=2)

axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('BW Utilization')
axes[1].set_title('Memory Bandwidth Utilization')
axes[1].legend(fontsize=7)
axes[1].grid(True, alpha=0.3)
axes[1].set_xscale('log', base=2)

axes[2].set_xlabel('Batch Size')
axes[2].set_ylabel('TPOT (ms)')
axes[2].set_title('Time per Output Token')
axes[2].legend(fontsize=7)
axes[2].grid(True, alpha=0.3)
axes[2].set_xscale('log', base=2)

plt.tight_layout()
plt.savefig('bandwidth_profile.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Software Stack Comparison

Different hardware requires different software stacks. Understanding kernel availability and maturity is critical.

In [ ]:
# Software stack feature comparison
software_features = {
    'CUDA (NVIDIA)': {
        'FlashAttention': 'Full (v2/v3)',
        'PagedAttention': 'Full (vLLM native)',
        'FP8 inference': 'Yes (H100+)',
        'INT4/GPTQ': 'Yes (ExLlama, AWQ)',
        'Continuous batching': 'Full',
        'Tensor Parallelism': 'Full (NCCL)',
        'Speculative decoding': 'Yes',
        'vLLM support': 'Tier 1',
        'SGLang support': 'Tier 1',
        'Custom kernel ease': 'Easy (Triton, CUDA)',
    },
    'ROCm (AMD)': {
        'FlashAttention': 'Yes (CK-based)',
        'PagedAttention': 'Yes (vLLM port)',
        'FP8 inference': 'Yes (MI300X)',
        'INT4/GPTQ': 'Partial',
        'Continuous batching': 'Full',
        'Tensor Parallelism': 'Full (RCCL)',
        'Speculative decoding': 'Yes',
        'vLLM support': 'Tier 2',
        'SGLang support': 'Tier 2',
        'Custom kernel ease': 'Moderate (Triton, HIP)',
    },
    'SynapseAI (Intel)': {
        'FlashAttention': 'Custom impl',
        'PagedAttention': 'Developing',
        'FP8 inference': 'Yes (Gaudi 2+)',
        'INT4/GPTQ': 'Limited',
        'Continuous batching': 'Developing',
        'Tensor Parallelism': 'Yes',
        'Speculative decoding': 'Planned',
        'vLLM support': 'Tier 3',
        'SGLang support': 'Experimental',
        'Custom kernel ease': 'Difficult (TPC)',
    },
    'JAX/XLA (Google TPU)': {
        'FlashAttention': 'Pallas impl',
        'PagedAttention': 'Custom impl',
        'FP8 inference': 'No',
        'INT4/GPTQ': 'Limited',
        'Continuous batching': 'Custom',
        'Tensor Parallelism': 'Yes (GSPMD)',
        'Speculative decoding': 'Yes',
        'vLLM support': 'Experimental (v0.6+)',
        'SGLang support': 'No',
        'Custom kernel ease': 'Moderate (Pallas)',
    },
}

# Print comparison table
features = list(next(iter(software_features.values())).keys())
print(f"{'Feature':>25}", end='')
for stack in software_features:
    print(f"{stack:>22}", end='')
print()
print("-" * (25 + 22 * len(software_features)))

for feature in features:
    print(f"{feature:>25}", end='')
    for stack, feats in software_features.items():
        print(f"{feats[feature]:>22}", end='')
    print()

## 5. Hardware-Specific Kernel Optimizations

In [ ]:
class KernelOptimizationAnalyzer:
    """Analyze kernel optimization opportunities for different hardware."""

    def __init__(self, hw: HardwareSpec):
        self.hw = hw

    def analyze_attention_optimization(self, seq_len: int, head_dim: int,
                                       num_heads: int) -> dict:
        """Analyze attention kernel optimization potential."""
        # Standard attention: materializes full attention matrix
        standard_mem = seq_len * seq_len * num_heads * 4  # FP32 attention matrix
        standard_flops = 4 * num_heads * seq_len * seq_len * head_dim  # QK^T + AV

        # FlashAttention: tile-based, no materialization
        flash_mem = num_heads * seq_len * head_dim * 2 * 2  # Only QKV+O in HBM
        flash_flops = standard_flops  # Same FLOPs, better memory access

        # Memory savings
        mem_savings = 1 - flash_mem / standard_mem

        # Throughput estimate
        standard_time = max(
            standard_flops / (self.hw.compute_fp16_tflops * 1e12),
            standard_mem / (self.hw.memory_bandwidth_gbps * 1e9)
        )
        flash_time = max(
            flash_flops / (self.hw.compute_fp16_tflops * 1e12),
            flash_mem / (self.hw.memory_bandwidth_gbps * 1e9)
        )

        return {
            'standard_mem_gb': standard_mem / 1e9,
            'flash_mem_gb': flash_mem / 1e9,
            'mem_savings': mem_savings,
            'standard_time_ms': standard_time * 1000,
            'flash_time_ms': flash_time * 1000,
            'speedup': standard_time / flash_time,
        }

    def analyze_quantization_impact(self, model_params_billions: float) -> dict:
        """Analyze impact of quantization on decode performance."""
        results = {}
        for dtype, bytes_per_param, compute_mult in [
            ('FP16', 2, 1.0),
            ('FP8', 1, 2.0 if self.hw.compute_fp8_tflops > 0 else 1.0),
            ('INT8', 1, 1.5),
            ('INT4', 0.5, 1.0),  # Usually dequantized for compute
        ]:
            model_bytes = model_params_billions * 1e9 * bytes_per_param
            can_fit = model_bytes / 1e9 < self.hw.memory_gb * 0.9

            # Decode time (memory-bound for bs=1)
            decode_time = model_bytes / (self.hw.memory_bandwidth_gbps * 1e9)

            effective_tflops = self.hw.compute_fp16_tflops * compute_mult

            results[dtype] = {
                'model_size_gb': model_bytes / 1e9,
                'can_fit_single': can_fit,
                'decode_tpot_ms': decode_time * 1000,
                'speedup_vs_fp16': (model_params_billions * 1e9 * 2 /
                                    (self.hw.memory_bandwidth_gbps * 1e9)) / decode_time,
            }
        return results


# Analyze optimizations across hardware
print(f"FlashAttention Speedup (seq_len=4096, 64 heads, head_dim=128):")
print(f"{'Hardware':>15} {'Std Mem (GB)':>12} {'Flash Mem':>10} {'Savings':>8} {'Speedup':>8}")
print("-" * 58)
for name, hw in list(hardware_db.items())[:6]:
    analyzer = KernelOptimizationAnalyzer(hw)
    result = analyzer.analyze_attention_optimization(4096, 128, 64)
    print(f"{hw.name:>15} {result['standard_mem_gb']:>10.2f}GB {result['flash_mem_gb']:>8.3f}GB "
          f"{result['mem_savings']:>7.0%} {result['speedup']:>7.1f}x")

print(f"\nQuantization Impact on LLaMA-70B Decode (batch_size=1):")
print(f"{'Hardware':>15} {'FP16 TPOT':>10} {'FP8 TPOT':>10} {'INT8 TPOT':>10} {'INT4 TPOT':>10} {'Fits FP16?':>10}")
print("-" * 70)
for name, hw in list(hardware_db.items())[:6]:
    analyzer = KernelOptimizationAnalyzer(hw)
    quant = analyzer.analyze_quantization_impact(70)
    print(f"{hw.name:>15}", end='')
    for dtype in ['FP16', 'FP8', 'INT8', 'INT4']:
        print(f"{quant[dtype]['decode_tpot_ms']:>8.1f}ms", end='')
    print(f"{'Yes' if quant['FP16']['can_fit_single'] else 'No':>10}")

---
## Assignment 1: Analyze Hardware-Specific Bottlenecks

In [ ]:
# ASSIGNMENT 1: Hardware Bottleneck Analysis

class BottleneckAnalyzer:
    """
    TODO: For a given workload, identify the primary bottleneck on each hardware.

    Bottleneck categories:
    1. Memory capacity (model doesn't fit)
    2. Memory bandwidth (decode bound)
    3. Compute (prefill bound)
    4. Interconnect (multi-GPU scaling)
    5. Software (missing kernel support)
    """

    def __init__(self, model_params_b: float, num_layers: int, hidden_dim: int):
        self.model_params = model_params_b
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

    def analyze(self, hw: HardwareSpec, batch_size: int = 1,
                seq_len: int = 2048, num_gpus: int = 1) -> dict:
        """
        TODO: Identify primary bottleneck and quantify impact.

        Returns:
        - primary_bottleneck: str
        - bottleneck_details: dict with utilization numbers
        - recommendations: list of optimization suggestions
        """
        # YOUR CODE HERE
        model_size_gb = self.model_params * 2  # FP16 GB
        kv_per_token = 2 * self.num_layers * 8 * 128 * 2  # Approximate
        kv_total_gb = kv_per_token * seq_len * batch_size / 1e9

        total_memory_needed = model_size_gb / num_gpus + kv_total_gb / num_gpus
        memory_utilization = total_memory_needed / hw.memory_gb

        # Compute utilization
        total_params = 12 * self.num_layers * self.hidden_dim ** 2
        prefill_flops = 2 * total_params * seq_len * batch_size
        prefill_time = prefill_flops / (hw.compute_fp16_tflops * 1e12 * num_gpus)

        # Decode bandwidth utilization
        decode_bytes = total_params * 2 / num_gpus
        decode_time = decode_bytes / (hw.memory_bandwidth_gbps * 1e9)

        recommendations = []
        if memory_utilization > 0.95:
            bottleneck = 'memory_capacity'
            recommendations.append('Use quantization (INT8/INT4) to reduce model size')
            recommendations.append(f'Use {int(np.ceil(total_memory_needed / (hw.memory_gb * 0.9)))} GPUs with tensor parallelism')
        elif batch_size <= 4:
            bottleneck = 'memory_bandwidth'
            recommendations.append('Increase batch size for better BW utilization')
            recommendations.append('Consider hardware with higher BW (MI300X, H200)')
        else:
            bottleneck = 'compute'
            recommendations.append('Consider FP8 inference for 2x compute')

        return {
            'primary_bottleneck': bottleneck,
            'memory_utilization': memory_utilization,
            'prefill_time_ms': prefill_time * 1000,
            'decode_tpot_ms': decode_time * 1000,
            'recommendations': recommendations,
        }


# Test bottleneck analysis
analyzer = BottleneckAnalyzer(model_params_b=70, num_layers=80, hidden_dim=8192)

for hw_name, hw in list(hardware_db.items())[:6]:
    result = analyzer.analyze(hw, batch_size=1, seq_len=2048, num_gpus=1)
    print(f"\n{hw.name}:")
    print(f"  Bottleneck: {result['primary_bottleneck']}")
    print(f"  Memory util: {result['memory_utilization']:.0%}")
    print(f"  Prefill time: {result['prefill_time_ms']:.1f}ms")
    print(f"  Decode TPOT: {result['decode_tpot_ms']:.1f}ms")
    for rec in result['recommendations']:
        print(f"  -> {rec}")

---
## Assignment 2: Adapt a Kernel for Different Hardware

In [ ]:
# ASSIGNMENT 2: Hardware-Adaptive Kernel Configuration

class AdaptiveKernelConfig:
    """
    TODO: Generate optimal kernel configurations for different hardware.

    For attention kernel, optimize:
    - Block sizes (tile M, tile N, tile K)
    - Number of warps/wavefronts
    - Shared memory usage
    - Pipeline depth
    """

    def __init__(self, hw: HardwareSpec):
        self.hw = hw

    def get_attention_config(self, seq_len: int, head_dim: int,
                              batch_heads: int) -> dict:
        """
        TODO: Return optimal tile sizes for attention kernel.
        """
        # YOUR CODE HERE
        if self.hw.vendor == 'NVIDIA':
            # CUDA: optimize for warp size 32
            if head_dim <= 64:
                block_m, block_n = 128, 64
            else:
                block_m, block_n = 64, 128
            num_warps = 4 if seq_len <= 2048 else 8
            num_stages = 3 if self.hw.compute_fp16_tflops > 500 else 2
        elif self.hw.vendor == 'AMD':
            # ROCm: optimize for wavefront size 64
            block_m, block_n = 64, 64
            num_warps = 4  # wavefronts
            num_stages = 2
        else:
            # Default conservative config
            block_m, block_n = 64, 64
            num_warps = 4
            num_stages = 1

        shared_mem_kb = (block_m + block_n) * head_dim * 2 * num_stages / 1024

        return {
            'BLOCK_M': block_m,
            'BLOCK_N': block_n,
            'num_warps': num_warps,
            'num_stages': num_stages,
            'shared_memory_kb': shared_mem_kb,
            'vendor_specific': self.hw.vendor,
        }

    def get_gemm_config(self, m: int, n: int, k: int) -> dict:
        """
        TODO: Return optimal GEMM tile configuration.
        """
        # YOUR CODE HERE
        if self.hw.vendor == 'NVIDIA':
            if m * n > 1e6:
                tile_m, tile_n, tile_k = 128, 256, 32
            else:
                tile_m, tile_n, tile_k = 64, 64, 32
        elif self.hw.vendor == 'AMD':
            tile_m, tile_n, tile_k = 128, 128, 16
        else:
            tile_m, tile_n, tile_k = 64, 64, 32

        return {
            'TILE_M': tile_m, 'TILE_N': tile_n, 'TILE_K': tile_k,
            'vendor': self.hw.vendor,
        }


# Test configurations
print(f"Attention Kernel Configs (seq=4096, head_dim=128):")
print(f"{'Hardware':>15} {'Block M':>8} {'Block N':>8} {'Warps':>6} {'Stages':>7} {'SMEM KB':>8}")
print("-" * 58)
for name, hw in list(hardware_db.items())[:6]:
    config = AdaptiveKernelConfig(hw)
    attn_cfg = config.get_attention_config(4096, 128, 64)
    print(f"{hw.name:>15} {attn_cfg['BLOCK_M']:>8} {attn_cfg['BLOCK_N']:>8} "
          f"{attn_cfg['num_warps']:>6} {attn_cfg['num_stages']:>7} {attn_cfg['shared_memory_kb']:>7.1f}")

---
## Assignment 3: Create a Hardware Recommendation Guide

In [ ]:
# ASSIGNMENT 3: Hardware Recommendation Engine

class HardwareRecommender:
    """
    TODO: Build a recommendation engine that suggests optimal hardware
    based on workload requirements.

    Input: model size, latency requirements, throughput needs, budget
    Output: ranked hardware recommendations with justification
    """

    def __init__(self, hardware_db: Dict[str, HardwareSpec]):
        self.db = hardware_db

    def recommend(self, model_params_b: float, max_latency_ms: float,
                  target_throughput: float, hourly_budget: float,
                  context_length: int = 4096,
                  quantization: str = 'FP16') -> List[dict]:
        """
        TODO: Return ranked list of hardware recommendations.

        Each recommendation should include:
        - hardware name
        - num_gpus needed
        - estimated latency
        - estimated throughput
        - total cost per hour
        - pros and cons
        - score (0-100)
        """
        # YOUR CODE HERE
        dtype_bytes = {'FP16': 2, 'FP8': 1, 'INT8': 1, 'INT4': 0.5}[quantization]
        model_size_gb = model_params_b * dtype_bytes

        recommendations = []
        for name, hw in self.db.items():
            # How many GPUs needed to fit model?
            num_gpus = max(1, int(np.ceil(model_size_gb / (hw.memory_gb * 0.8))))

            # Decode TPOT
            total_params = model_params_b * 1e9
            decode_bytes = total_params * dtype_bytes / num_gpus
            tpot = decode_bytes / (hw.memory_bandwidth_gbps * 1e9) * 1000  # ms

            # Throughput estimate
            opt_batch = max(1, int(hw.memory_bandwidth_gbps / (hw.compute_fp16_tflops * 1e3 / total_params)))
            throughput = opt_batch / (tpot / 1000)

            # Cost
            total_cost = hw.price_per_hour * num_gpus

            # Score (multi-criteria)
            latency_score = max(0, 1 - tpot / max_latency_ms) * 30
            throughput_score = min(30, throughput / target_throughput * 30)
            cost_score = max(0, 1 - total_cost / hourly_budget) * 20
            maturity_bonus = {'mature': 20, 'developing': 10, 'early': 5}[hw.maturity]
            total_score = latency_score + throughput_score + cost_score + maturity_bonus

            # Pros and cons
            pros, cons = [], []
            if hw.memory_gb >= 128:
                pros.append('Large memory for big models')
            if hw.memory_bandwidth_gbps > 3000:
                pros.append('High bandwidth for fast decode')
            if hw.maturity != 'mature':
                cons.append(f'Software stack: {hw.maturity}')
            if total_cost > hourly_budget:
                cons.append('Exceeds budget')
            if not pros:
                pros.append('Cost-effective option')

            recommendations.append({
                'hardware': hw.name,
                'vendor': hw.vendor,
                'num_gpus': num_gpus,
                'tpot_ms': tpot,
                'throughput': throughput,
                'cost_per_hour': total_cost,
                'score': total_score,
                'pros': pros,
                'cons': cons,
            })

        return sorted(recommendations, key=lambda r: r['score'], reverse=True)


# Test recommendations
recommender = HardwareRecommender(hardware_db)

scenarios = [
    ('Chatbot (7B)', 7, 50, 100, 10, 4096, 'FP16'),
    ('Enterprise (70B)', 70, 100, 20, 30, 4096, 'FP16'),
    ('Budget (70B INT4)', 70, 200, 10, 15, 4096, 'INT4'),
]

for desc, params, lat, tp, budget, ctx, quant in scenarios:
    print(f"\n{'='*70}")
    print(f" Scenario: {desc}")
    print(f" Model: {params}B, Latency SLO: {lat}ms, Throughput: {tp} tok/s, Budget: ${budget}/hr")
    print(f"{'='*70}")

    recs = recommender.recommend(params, lat, tp, budget, ctx, quant)
    for i, r in enumerate(recs[:5]):
        print(f"  #{i+1} {r['hardware']:>15} ({r['vendor']}) | {r['num_gpus']} GPUs | "
              f"TPOT={r['tpot_ms']:.1f}ms | ${r['cost_per_hour']:.2f}/hr | "
              f"Score={r['score']:.0f}")
        print(f"     Pros: {', '.join(r['pros'])}")
        if r['cons']:
            print(f"     Cons: {', '.join(r['cons'])}")

## Summary

### Key Takeaways

1. **Memory bandwidth** is the primary bottleneck for LLM decode -- hardware with higher bandwidth (MI300X, H200) offers better per-token latency

2. **Memory capacity** determines the largest model you can serve -- MI300X (192GB) enables single-chip 70B serving

3. **Software maturity** matters as much as raw hardware specs -- CUDA ecosystem advantage is significant

4. **Hardware selection** depends on the specific workload: latency-critical uses need high BW, throughput uses need high compute

5. **Quantization** is a powerful equalizer -- INT4 can make budget hardware competitive with premium options

### Further Reading
- AMD ROCm for vLLM: https://docs.vllm.ai/en/latest/getting_started/amd-installation.html
- Intel Gaudi: https://docs.habana.ai/
- Google TPU: https://cloud.google.com/tpu/docs